## CSCI612 Computer Vision Model to Detect and Classify Cars, Buses, Trucks, Pedestrians, Motorcycles, and Bicycles.
### 1) How to convert json label from the BDD400K data set to YOLO compatible labels.

# Checking for bad labels.

In [2]:
from pathlib import Path

label_dirs = [
    Path("bdd_yolo_6cls/labels/train"),
    Path("bdd_yolo_6cls/labels/val"),
]

bad = []

for label_dir in label_dirs:
    for txt in label_dir.glob("*.txt"):
        for i, line in enumerate(txt.read_text().splitlines(), start=1):
            if not line.strip():
                continue

            parts = line.split()
            cls = int(float(parts[0]))

            if cls < 0 or cls > 5:
                bad.append((txt, i, cls, line))

print(f"Bad labels found: {len(bad)}")

for item in bad[:20]:
    print(item)

Bad labels found: 0


# Revaldiate the labels before we can blame mps for the errors

In [2]:
from pathlib import Path
import math

DATASET_ROOT = Path("bdd_yolo_6cls")
NUM_CLASSES = 6

label_dirs = [
    DATASET_ROOT / "labels" / "train",
    DATASET_ROOT / "labels" / "val",
]

issues = []

for label_dir in label_dirs:
    for txt_file in sorted(label_dir.glob("*.txt")):
        lines = txt_file.read_text().splitlines()

        for line_num, line in enumerate(lines, start=1):
            line = line.strip()

            # Empty label files are valid
            if not line:
                continue

            parts = line.split()

            if len(parts) != 5:
                issues.append((txt_file, line_num, "wrong_number_of_values", line))
                continue

            try:
                cls = int(parts[0])
                x, y, w, h = map(float, parts[1:])
            except ValueError:
                issues.append((txt_file, line_num, "non_numeric_value", line))
                continue

            if cls < 0 or cls >= NUM_CLASSES:
                issues.append((txt_file, line_num, "invalid_class_id", line))

            values = [x, y, w, h]

            if any(math.isnan(v) or math.isinf(v) for v in values):
                issues.append((txt_file, line_num, "nan_or_inf", line))
                continue

            if not (0 <= x <= 1):
                issues.append((txt_file, line_num, "x_center_out_of_range", line))

            if not (0 <= y <= 1):
                issues.append((txt_file, line_num, "y_center_out_of_range", line))

            if not (0 < w <= 1):
                issues.append((txt_file, line_num, "width_out_of_range", line))

            if not (0 < h <= 1):
                issues.append((txt_file, line_num, "height_out_of_range", line))

            x1 = x - w / 2
            y1 = y - h / 2
            x2 = x + w / 2
            y2 = y + h / 2

            if x1 < 0 or y1 < 0 or x2 > 1 or y2 > 1:
                issues.append((txt_file, line_num, "box_extends_outside_image", line))


print(f"Validation complete.")
print(f"Total issues found: {len(issues)}")

for issue in issues[:50]:
    print(issue)

if len(issues) > 50:
    print(f"... showing first 50 of {len(issues)} issues")

Validation complete.
Total issues found: 8976
(PosixPath('bdd_yolo_6cls/labels/train/00067cfb-5adfaaa7.txt'), 7, 'box_extends_outside_image', '0 0.920083 0.556720 0.159835 0.160079')
(PosixPath('bdd_yolo_6cls/labels/train/00091078-7cff8ea6.txt'), 2, 'box_extends_outside_image', '0 0.076537 0.611250 0.153075 0.225000')
(PosixPath('bdd_yolo_6cls/labels/train/00091078-cedbfea7.txt'), 6, 'box_extends_outside_image', '1 0.743309 0.499110 0.510671 0.998221')
(PosixPath('bdd_yolo_6cls/labels/train/00134776-9123d227.txt'), 3, 'box_extends_outside_image', '0 0.919682 0.639455 0.160637 0.368957')
(PosixPath('bdd_yolo_6cls/labels/train/00225f53-4200bde2.txt'), 12, 'box_extends_outside_image', '0 0.036083 0.622651 0.072167 0.162250')
(PosixPath('bdd_yolo_6cls/labels/train/00268999-0b20ef00.txt'), 1, 'box_extends_outside_image', '0 0.048778 0.323255 0.097557 0.094237')
(PosixPath('bdd_yolo_6cls/labels/train/002827ad-8748e2fb.txt'), 8, 'box_extends_outside_image', '0 0.010235 0.300674 0.020471 0.029

# New way to generate labels.
Starting from scratch delete previous generated YOLO labels

In [3]:
from shutil import rmtree

rmtree("bdd_yolo_6cls/labels/train")
rmtree("bdd_yolo_6cls/labels/val")

In [7]:
from pathlib import Path
from shutil import rmtree
from collections import Counter
import json
import math


# ----------------------------
# Configuration
# ----------------------------

TARGET_CLASSES = [
    "car",
    "bus",
    "truck",
    "pedestrian",
    "motorcycle",
    "bicycle",
]

DATASET_ROOT = Path("bdd_yolo_6cls")
NUM_CLASSES = len(TARGET_CLASSES)

TRAIN_JSON = "archive/labels/det_v2_train_release.json"
VAL_JSON = "archive/labels/det_v2_val_release.json"

TRAIN_LABEL_DIR = DATASET_ROOT / "labels" / "train"
VAL_LABEL_DIR = DATASET_ROOT / "labels" / "val"


# ----------------------------
# Convert BDD box to YOLO box
# ----------------------------

def convert_box(box, img_w=1280, img_h=720):
    x1, y1 = box["x1"], box["y1"]
    x2, y2 = box["x2"], box["y2"]

    # Clip box to image boundaries
    x1 = max(0, min(x1, img_w))
    y1 = max(0, min(y1, img_h))
    x2 = max(0, min(x2, img_w))
    y2 = max(0, min(y2, img_h))

    # Skip invalid boxes
    if x2 <= x1 or y2 <= y1:
        return None

    x_center = ((x1 + x2) / 2) / img_w
    y_center = ((y1 + y2) / 2) / img_h
    width = (x2 - x1) / img_w
    height = (y2 - y1) / img_h

    return x_center, y_center, width, height


# ----------------------------
# Convert BDD JSON to YOLO labels
# ----------------------------

def convert_bdd_json(json_path, output_label_dir, target_classes, clean_output=True):
    output_label_dir = Path(output_label_dir)

    if clean_output and output_label_dir.exists():
        rmtree(output_label_dir)

    output_label_dir.mkdir(parents=True, exist_ok=True)

    class_map = {name: idx for idx, name in enumerate(target_classes)}

    with open(json_path, "r") as f:
        data = json.load(f)

    total_images = 0
    total_boxes = 0
    empty_labels = 0

    for item in data:
        total_images += 1

        image_name = item["name"]
        label_path = output_label_dir / f"{Path(image_name).stem}.txt"

        lines = []

        for obj in item.get("labels") or []:
            category = obj.get("category")

            if category not in class_map:
                continue

            if "box2d" not in obj:
                continue

            box = convert_box(obj["box2d"])

            if box is None:
                continue

            x, y, w, h = box
            class_id = class_map[category]

            lines.append(f"{class_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")

        if len(lines) == 0:
            empty_labels += 1

        total_boxes += len(lines)
        label_path.write_text("\n".join(lines))

    print(f"Converted: {json_path}")
    print(f"  Images processed: {total_images}")
    print(f"  YOLO boxes written: {total_boxes}")
    print(f"  Empty label files: {empty_labels}")
    print(f"  Output folder: {output_label_dir}")


# ----------------------------
# Validate YOLO labels
# ----------------------------

def validate_yolo_labels(label_dirs, num_classes):
    issues = []
    issue_counts = Counter()
    total_files = 0
    total_boxes = 0
    empty_files = 0

    for label_dir in label_dirs:
        for txt_file in sorted(label_dir.glob("*.txt")):
            total_files += 1
            lines = txt_file.read_text().splitlines()

            if len(lines) == 0:
                empty_files += 1
                continue

            for line_num, line in enumerate(lines, start=1):
                line = line.strip()

                if not line:
                    continue

                parts = line.split()

                if len(parts) != 5:
                    issue_type = "wrong_number_of_values"
                    issues.append((txt_file, line_num, issue_type, line))
                    issue_counts[issue_type] += 1
                    continue

                try:
                    cls = int(parts[0])
                    x, y, w, h = map(float, parts[1:])
                except ValueError:
                    issue_type = "non_numeric_value"
                    issues.append((txt_file, line_num, issue_type, line))
                    issue_counts[issue_type] += 1
                    continue

                total_boxes += 1

                if cls < 0 or cls >= num_classes:
                    issue_type = "invalid_class_id"
                    issues.append((txt_file, line_num, issue_type, line))
                    issue_counts[issue_type] += 1

                values = [x, y, w, h]

                if any(math.isnan(v) or math.isinf(v) for v in values):
                    issue_type = "nan_or_inf"
                    issues.append((txt_file, line_num, issue_type, line))
                    issue_counts[issue_type] += 1
                    continue

                checks = [
                    ("x_center_out_of_range", not (0 <= x <= 1)),
                    ("y_center_out_of_range", not (0 <= y <= 1)),
                    ("width_out_of_range", not (0 < w <= 1)),
                    ("height_out_of_range", not (0 < h <= 1)),
                ]

                for issue_type, failed in checks:
                    if failed:
                        issues.append((txt_file, line_num, issue_type, line))
                        issue_counts[issue_type] += 1

                x1 = x - w / 2
                y1 = y - h / 2
                x2 = x + w / 2
                y2 = y + h / 2

                tolerance = 1e-6

                if (
                    x1 < -tolerance
                    or y1 < -tolerance
                    or x2 > 1 + tolerance
                    or y2 > 1 + tolerance
                ):
                    issue_type = "box_extends_outside_image"
                    issues.append((txt_file, line_num, issue_type, line))
                    issue_counts[issue_type] += 1

    return issues, issue_counts, total_files, total_boxes, empty_files


# ----------------------------
# Run conversion
# ----------------------------

convert_bdd_json(
    json_path=TRAIN_JSON,
    output_label_dir=TRAIN_LABEL_DIR,
    target_classes=TARGET_CLASSES,
)

convert_bdd_json(
    json_path=VAL_JSON,
    output_label_dir=VAL_LABEL_DIR,
    target_classes=TARGET_CLASSES,
)


# ----------------------------
# Delete old Ultralytics cache files
# ----------------------------

for cache_file in [
    DATASET_ROOT / "labels" / "train.cache",
    DATASET_ROOT / "labels" / "val.cache",
]:
    if cache_file.exists():
        cache_file.unlink()
        print(f"Deleted cache file: {cache_file}")


# ----------------------------
# Run validation
# ----------------------------

LABEL_DIRS = [
    TRAIN_LABEL_DIR,
    VAL_LABEL_DIR,
]

issues, issue_counts, total_files, total_boxes, empty_files = validate_yolo_labels(
    LABEL_DIRS,
    NUM_CLASSES,
)

print("\nValidation complete.")
print(f"Total label files checked: {total_files}")
print(f"Total bounding boxes checked: {total_boxes}")
print(f"Empty label files: {empty_files}")
print(f"Total issues found: {len(issues)}")

print("\nIssue summary:")
for issue_type, count in issue_counts.items():
    print(f"  {issue_type}: {count}")

print("\nFirst 50 issues:")
for issue in issues[:50]:
    print(issue)

if len(issues) > 50:
    print(f"\n... showing first 50 of {len(issues)} issues")

Converted: archive/labels/det_v2_train_release.json
  Images processed: 69863
  YOLO boxes written: 842878
  Empty label files: 585
  Output folder: bdd_yolo_6cls/labels/train
Converted: archive/labels/det_v2_val_release.json
  Images processed: 10000
  YOLO boxes written: 123664
  Empty label files: 81
  Output folder: bdd_yolo_6cls/labels/val
Deleted cache file: bdd_yolo_6cls/labels/train.cache
Deleted cache file: bdd_yolo_6cls/labels/val.cache

Validation complete.
Total label files checked: 79863
Total bounding boxes checked: 966542
Empty label files: 666
Total issues found: 0

Issue summary:

First 50 issues:


## Training module

In [ ]:
from ultralytics import YOLO
from datetime import datetime
from pathlib import Path

model = YOLO("yolo11n.pt")

start_time = datetime.now()
print(f"Training started at: {start_time:%Y-%m-%d %H:%M:%S}")

results = model.train(
    data="bdd_yolo_6cls/data.yaml",
    epochs=2,
    imgsz=640,
    batch=8,
    device="mps",
    workers=5,
    cache=False,
    amp=False,
    close_mosaic=0,
    plots=True,
)

finish_time = datetime.now()
elapsed_time = finish_time - start_time

print(f"Training finished at: {finish_time:%Y-%m-%d %H:%M:%S}")
print(f"Elapsed training time: {elapsed_time}")

Path("training_time_log.txt").write_text(
    f"Training started at: {start_time:%Y-%m-%d %H:%M:%S}\n"
    f"Training finished at: {finish_time:%Y-%m-%d %H:%M:%S}\n"
    f"Elapsed training time: {elapsed_time}\n"
)

Training started at: 2026-07-03 10:59:31
New https://pypi.org/project/ultralytics/8.4.87 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.47 🚀 Python-3.11.15 torch-2.11.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=bdd_yolo_6cls/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=2, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scal

## Make sure we have the right torchvision for the training

In [1]:
%pip install torchvision==0.25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 34.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.4/79.4 MB 40.6 MB/s  0:00:01a 0:00:010:00:01:01
  Attempting uninstall: torch
    Found existing installation: torch 2.11.0
    Uninstalling torch-2.11.0:━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [torch]
      Successfully uninstalled torch-2.11.0━━━━━━━ 0/2 [torch]
  Attempting uninstall: torchvision━━━━━━━━━━━━━━━━━━━ 0/2 [torch]
    Found existing installation: torchvision 0.26.0 0/2 [torch]
    Uninstalling torchvision-0.26.0:━━━━━━━━━━━━━━ 0/2 [torch]
      Successfully uninstalled torchvision-0.26.0━ 0/2 [torch]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torchvision] 1/2 [torchvision]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)

2.10.0
0.25.0


## Create a balanced subsetof only 10K images

In [3]:
from pathlib import Path
from shutil import copy2, rmtree
from collections import defaultdict
import random

# ----------------------------
# Configuration
# ----------------------------

SRC = Path("/Users/car/612/A5/YOLOv11/bdd_yolo_6cls")
DST = Path("/Users/car/612/A5/YOLO10K/bdd_yolo_6cls")

CLASS_NAMES = [
    "car",
    "bus",
    "truck",
    "pedestrian",
    "motorcycle",
    "bicycle",
]

NUM_CLASSES = len(CLASS_NAMES)

SPLIT_TOTALS = {
    "train": 7000,
    "val": 1000,
    "test": 2000,
}

SEED = 42
random.seed(SEED)


# ----------------------------
# Helper: balanced quota
# ----------------------------

def make_balanced_quota(total, num_classes):
    base = total // num_classes
    remainder = total % num_classes

    quota = {}
    for cls_id in range(num_classes):
        quota[cls_id] = base + (1 if cls_id < remainder else 0)

    return quota


# ----------------------------
# Collect all valid image/label pairs
# ----------------------------

image_by_stem = {}

for split in ["train", "val"]:
    image_dir = SRC / "images" / split
    label_dir = SRC / "labels" / split

    images = list(image_dir.glob("*.jpg")) + list(image_dir.glob("*.png"))

    for img in images:
        label = label_dir / f"{img.stem}.txt"

        if label.exists() and label.read_text().strip():
            image_by_stem[img.stem] = {
                "image": img,
                "label": label,
            }


print(f"Total non-empty image/label pairs found: {len(image_by_stem)}")


# ----------------------------
# Build class-to-images mapping
# ----------------------------

class_to_stems = defaultdict(list)

for stem, paths in image_by_stem.items():
    label_path = paths["label"]
    classes_in_image = set()

    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue

        cls_id = int(float(parts[0]))

        if 0 <= cls_id < NUM_CLASSES:
            classes_in_image.add(cls_id)

    for cls_id in classes_in_image:
        class_to_stems[cls_id].append(stem)


for cls_id, name in enumerate(CLASS_NAMES):
    print(f"{name}: {len(class_to_stems[cls_id])} candidate images")


# ----------------------------
# Create balanced splits
# ----------------------------

used_stems = set()
split_assignments = {
    "train": [],
    "val": [],
    "test": [],
}

for split, total in SPLIT_TOTALS.items():
    quota = make_balanced_quota(total, NUM_CLASSES)

    print(f"\nCreating {split} split")
    print("Quota:", {CLASS_NAMES[k]: v for k, v in quota.items()})

    for cls_id, needed in quota.items():
        candidates = class_to_stems[cls_id].copy()
        random.shuffle(candidates)

        selected = []

        for stem in candidates:
            if stem in used_stems:
                continue

            selected.append(stem)
            used_stems.add(stem)

            if len(selected) == needed:
                break

        if len(selected) < needed:
            raise RuntimeError(
                f"Not enough images for class {CLASS_NAMES[cls_id]} in {split}. "
                f"Needed {needed}, found {len(selected)}."
            )

        split_assignments[split].extend(selected)

    random.shuffle(split_assignments[split])
    print(f"{split}: selected {len(split_assignments[split])} images")


# ----------------------------
# Copy files
# ----------------------------

if DST.exists():
    rmtree(DST)

for split, stems in split_assignments.items():
    dst_img_dir = DST / "images" / split
    dst_lbl_dir = DST / "labels" / split

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    for stem in stems:
        img = image_by_stem[stem]["image"]
        label = image_by_stem[stem]["label"]

        copy2(img, dst_img_dir / img.name)
        copy2(label, dst_lbl_dir / label.name)

    print(f"{split}: copied {len(stems)} image/label pairs")


# ----------------------------
# Write data.yaml
# ----------------------------

yaml_lines = [
    f"path: {DST}",
    "train: images/train",
    "val: images/val",
    "test: images/test",
    "",
    f"nc: {NUM_CLASSES}",
    "names:",
]

for idx, name in enumerate(CLASS_NAMES):
    yaml_lines.append(f"  {idx}: {name}")

(DST / "data.yaml").write_text("\n".join(yaml_lines))

print("\nBalanced dataset created successfully.")
print(f"Dataset path: {DST}")
print(f"YAML path: {DST / 'data.yaml'}")

Total non-empty image/label pairs found: 79197
car: 78825 candidate images
bus: 10580 candidate images
truck: 20811 candidate images
pedestrian: 25730 candidate images
motorcycle: 2673 candidate images
bicycle: 4990 candidate images

Creating train split
Quota: {'car': 1167, 'bus': 1167, 'truck': 1167, 'pedestrian': 1167, 'motorcycle': 1166, 'bicycle': 1166}
train: selected 7000 images

Creating val split
Quota: {'car': 167, 'bus': 167, 'truck': 167, 'pedestrian': 167, 'motorcycle': 166, 'bicycle': 166}
val: selected 1000 images

Creating test split
Quota: {'car': 334, 'bus': 334, 'truck': 333, 'pedestrian': 333, 'motorcycle': 333, 'bicycle': 333}
test: selected 2000 images
train: copied 7000 image/label pairs
val: copied 1000 image/label pairs
test: copied 2000 image/label pairs

Balanced dataset created successfully.
Dataset path: /Users/car/612/A5/YOLO10K/bdd_yolo_6cls
YAML path: /Users/car/612/A5/YOLO10K/bdd_yolo_6cls/data.yaml


# Training Module

In [2]:
from ultralytics import YOLO
from datetime import datetime
from pathlib import Path

model = YOLO("yolo11n.pt")

start_time = datetime.now()
print(f"Training started at: {start_time:%Y-%m-%d %H:%M:%S}")
results = model.train(
    data="bdd_yolo_6cls/data.yaml",
    epochs=10,
    imgsz=640,
    batch=8,
    device="mps",
    workers=4,
    cache=False,
    amp=False,
    close_mosaic=0,
    val=False,
    plots=False,
    save_period=1,
)

finish_time = datetime.now()
elapsed_time = finish_time - start_time

print(f"Training finished at: {finish_time:%Y-%m-%d %H:%M:%S}")
print(f"Elapsed training time: {elapsed_time}")

Path("training_time_log.txt").write_text(
    f"Training started at: {start_time:%Y-%m-%d %H:%M:%S}\n"
    f"Training finished at: {finish_time:%Y-%m-%d %H:%M:%S}\n"
    f"Elapsed training time: {elapsed_time}\n"
)

Training started at: 2026-07-04 07:54:53
New https://pypi.org/project/ultralytics/8.4.87 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.47 🚀 Python-3.11.15 torch-2.10.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=bdd_yolo_6cls/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_sca

RuntimeError: The size of tensor a (11338) must match the size of tensor b (9314) at non-singleton dimension 0

This is actually a very important clue, and it may explain some of the instability you’ve been seeing.

The key message is:
The volume "Macintosh HD" is out of space.

This is not a YOLO error and not a dataset error. It’s macOS telling you that the MPS (Metal Performance Shaders) runtime is trying to create temporary files (mpsgraph-...) and the disk is full.

Why this matters

During training, YOLO writes:

* checkpoints (best.pt, last.pt)
* plots (results.png, confusion_matrix.png, etc.)
* logs
* cache files

At the same time, MPS and PyTorch may create temporary files and caches.

If your disk fills up:

* MPS may fail unpredictably.
* Checkpoints may not be written correctly.
* You can see seemingly unrelated runtime errors because internal operations fail.

First thing to check

Run in Terminal: df -h /
Or df -h

This will show how much free space is left on your system volume.

You can also check from Python:

In [3]:
import shutil

total, used, free = shutil.disk_usage("/")

print(f"Total: {total / 1024**3:.1f} GB")
print(f"Used : {used / 1024**3:.1f} GB")
print(f"Free : {free / 1024**3:.1f} GB")

Total: 228.3 GB
Used : 222.6 GB
Free : 5.7 GB


What I’d recommend

For deep learning training, I generally like to have at least:

* 20–30 GB free as a minimum.
* 50 GB or more is much more comfortable, especially with large datasets and checkpoints.

Clean up candidates

You can often recover space by removing:

* Old runs/ directories from previous YOLO experiments.
* Old datasets you no longer need.
* Unused Conda environments.
* Downloads and large videos.

For example, if you no longer need earlier runs:
rm -rf runs/

or remove only older experiment folders.

I think this changes our diagnosis

Earlier we concluded the MPS backend was unstable. I still think MPS has limitations, but running out of disk space introduces another source of failures.

So before doing more training, I’d:

1. Free up a significant amount of disk space.
2. Reboot the Mac (to clear MPS state and temporary files).
3. Retry the 10K training.

One question: How much free disk space do you currently have? If you run the shutil.disk_usage("/") snippet above and share the output, we can see whether disk space is likely contributing to the crashes.